# 🛠️ Section 5 — Pandas : Découverte des DataFrames, Nettoyage et Analyse de la Consommation Énergétique

Bienvenue dans le cinquième module d'outils du cours **Data Science & Python** !

Dans les **Sections 1 & 2** (NumPy) et **Sections 3 & 4** (Perceptron & Descente de Gradient), nous avons exploré les fondations du calcul matriciel et de l'optimisation d'algorithmes.

NumPy est exceptionnellement rapide et puissant pour manipuler des matrices de nombres homogènes (uniquement des flottants ou uniquement des entiers). Cependant, dans la vie réelle d'un Data Scientist, les données ne sont presque jamais de simples blocs de nombres bruts :
- Elles comportent des **dates et des heures** (*timestamps*).
- Elles associent du **texte** (catégories, noms de capteurs, états de fonctionnement) et des **nombres** (puissance, tension, température).
- Elles possèdent des **noms de colonnes** explicites et un **index**.
- Elles contiennent fréquemment des **données manquantes** ou corrompues.

C'est précisément la raison d'être de **Pandas**, la bibliothèque standard mondiale pour la manipulation et l'analyse exploratoire de données tabulaires (*Exploratory Data Analysis — EDA*).

---

### 🏠 Le Cas d'Étude : La Consommation Électrique d'une Maison Connectée
Dans ce module, vous découvrez Pandas de zéro à travers un cas concret et moderne : **l'audit énergétique d'un foyer individuel équipé de capteurs connectés (IoT / Linky)**.

Pendant deux semaines, les capteurs ont enregistré toutes les 10 secondes :
- L'horodatage précis du relevé
- La **puissance active** appelée par la maison (en Watts)
- La **tension** du réseau électrique (en Volts)
- La **température extérieure** (°C)
- Le **circuit dominant** en activité (cuisine, buanderie, salon, veille)
- La **période tarifaire** en vigueur (Heures Pleines / Heures Creuses)
- Le **statut du capteur** (fonctionnement normal ou perte de transmission)

---

### 🗺️ Votre parcours d'apprentissage dans ce notebook :
0. **👁️ Coup d'œil visuel initial** — Le pouls électrique de la maison en un coup d'œil
1. **Section 1** — Cadrage du problème, Objectifs & Échelles de mesure
2. **Section 2** — Découverte de Pandas : Series, DataFrames & Diagnostic initial
3. **Section 3** — Nettoyage & Gestion des Données Manquantes (`NaN`)
4. **Section 4** — Feature Engineering : Créer de nouvelles colonnes métier ($kW$, $kWh$, catégories, €)
5. **Section 5** — DatetimeIndex & Slicing Temporel Express
6. **Section 6** — Rééchantillonnage temporel (`resample`) & Bilans Journaliers
7. **Section 7** — Fenêtres Glissantes (`rolling`) & Détection du Talon de Veille
8. **Section 8** — Profilage Métier & Tableau de Bord Visuel
9. **Section 9** — 🏆 Défi Final : L'Audit Énergétique de la Maison


In [ ]:
# Programme 1 — Initialisation & Outils interactifs (QCM et validation)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random as _random_qcm
import ipywidgets as w
from IPython.display import display

# Configuration graphique soignée
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (11, 4.5)
plt.rcParams['font.size'] = 10

# Relances humoristiques adaptées à l'univers Pandas & Data Science
_RELANCES_HUMOUR = [
    "SettingWithCopyWarning dans ta logique ! Vérifie si tu as fait une copie ou une vue.",
    "Attention à l'alignement des index Pandas : deux séries ne s'additionnent pas si leurs index divergent !",
    "Un NaN indésirable s'est infiltré dans ton calcul !",
    "Attention à la fréquence de rééchantillonnage : '1h' pour une heure, '1D' pour un jour !",
    "Ne confonds pas .loc (par libellé/date) et .iloc (par position entière) !",
    "Même Wes McKinney (le créateur de Pandas) a dû relire la doc de groupby un jour !"
]

def qcm(question, options, correct, explication=""):
    """
    Affiche un QCM interactif avec ipywidgets.
    correct : index (int) ou liste d'index de la/des bonne(s) réponse(s).
    """
    if isinstance(correct, int):
        correct = [correct]

    boutons = w.RadioButtons(options=options, layout=w.Layout(width='100%'))
    sortie  = w.Output()
    titre   = w.HTML(f"<b>❓ {question}</b>")

    def verifier(change):
        idx = options.index(change['new'])
        sortie.clear_output()
        with sortie:
            if idx in correct:
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#d4edda;color:#155724;border-left:5px solid #28a745'>"
                    f"✅ <b>Bravo, c'est exact !</b><br>{explication}</div>"
                ))
            else:
                humour = _random_qcm.choice(_RELANCES_HUMOUR)
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#f8d7da;color:#721c24;border-left:5px solid #dc3545'>"
                    f"❌ <b>Pas tout à fait...</b> {humour}<br><br>"
                    f"<b>Explication :</b> {explication}</div>"
                ))

    boutons.observe(verifier, names='value')
    display(w.VBox([titre, boutons, sortie]))

def check(nom, valeur, attendu, tol=1e-4):
    """
    Vérifie la valeur d'une variable, Series ou DataFrame dans un exercice.
    Affiche un retour visuel vert ou rouge.
    """
    ok = False
    try:
        if isinstance(attendu, (pd.Series, pd.DataFrame)) and isinstance(valeur, (pd.Series, pd.DataFrame)):
            if attendu.shape == valeur.shape:
                if isinstance(attendu, pd.Series):
                    ok = bool(np.allclose(valeur.dropna().values, attendu.dropna().values, atol=tol))
                else:
                    ok = list(valeur.columns) == list(attendu.columns)
        elif isinstance(attendu, np.ndarray) and isinstance(valeur, np.ndarray):
            ok = valeur.shape == attendu.shape and bool(np.allclose(valeur, attendu, atol=tol))
        elif isinstance(attendu, (int, float, np.integer, np.floating)) and isinstance(valeur, (int, float, np.integer, np.floating)):
            ok = abs(float(valeur) - float(attendu)) <= tol
        elif isinstance(attendu, (tuple, list)) and isinstance(valeur, (tuple, list)):
            ok = len(valeur) == len(attendu) and all(
                (abs(float(v) - float(a)) <= tol if isinstance(a, (int, float)) else v == a)
                for v, a in zip(valeur, attendu)
            )
        else:
            ok = bool(valeur == attendu)
    except Exception:
        ok = False

    if ok:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#d4edda;color:#155724;border-left:4px solid #28a745'>"
            f"✅ <b>{nom}</b> est correct ! (Valeur : <code>{repr(valeur)}</code>)</div>"
        ))
    else:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#f8d7da;color:#721c24;border-left:4px solid #dc3545'>"
            f"❌ <b>{nom}</b> n'est pas encore la valeur attendue.<br>"
            f"Ta valeur : <code>{repr(valeur)}</code><br>"
            f"Valeur attendue : <code>{repr(attendu)}</code></div>"
        ))

print(f"✅ Environnement initialisé avec Pandas version {pd.__version__} et NumPy {np.__version__}")
print("✅ Fonctions qcm() et check() prêtes à l'emploi.")


---
## 👁️ Coup d'œil visuel initial : Le pouls électrique de la maison

Avant même de disséquer le code et d'apprendre les commandes de Pandas, **regardons à quoi ressemble le signal que nous allons manipuler**. 
Un bon Data Scientist commence toujours par observer visuellement son terrain d'exploration !

Exécutez la cellule ci-dessous pour afficher :
1. **En haut** : La vue d'ensemble des **2 semaines complètes** (~100 000 mesures à raison d'une toutes les 10 secondes).
2. **En bas à gauche** : Un zoom sur une journée de **24 heures** (Lundi 22 juin), révélant le cycle de vie du foyer (le calme de la nuit, le réveil matinal, et les pointes de cuisine).
3. **En bas à droite** : La température extérieure synchronisée sur cette même journée.


In [ ]:
# Programme 2 — Visualisation initiale : Le pouls électrique de la maison
# Chargement direct pour donner une première intuition visuelle
_df_init = pd.read_csv("data/conso_maison_capteurs.csv")
_df_init['ts'] = pd.to_datetime(_df_init['timestamp'])
_df_init = _df_init.set_index('ts')

fig = plt.figure(figsize=(14, 8.5))
gs = fig.add_gridspec(2, 2, height_ratios=[1.2, 1])
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

# 1. Vue globale sur 2 semaines complètes
ax1.plot(_df_init.index, _df_init['puissance_W'], color='#1d3557', alpha=0.75, linewidth=0.8, label='Puissance active (W)')
ax1.axhline(6000, color='#e63946', linestyle='--', linewidth=2, label='Seuil disjoncteur 6 kVA (6 000 W)')
ax1.axhline(510, color='#2a9d8f', linestyle=':', linewidth=2, label='Talon de veille permanent (~510 W)')
ax1.set_title("1. Vue globale : 2 semaines complètes de relevés de la maison (99 396 points à 10 s d'intervalle)", fontweight='bold')
ax1.set_ylabel("Puissance (Watts)")
ax1.set_ylim(-100, 7500)
ax1.legend(loc='upper right')

# 2. Zoom sur 24 heures (Lundi 22 juin)
zoom_jour = _df_init.loc['2026-06-22']
ax2.plot(zoom_jour.index, zoom_jour['puissance_W'], color='#e76f51', linewidth=1.2, label='Puissance (W)')
ax2.axhline(510, color='#2a9d8f', linestyle=':', linewidth=1.5, label='Talon de veille')
# Coloration des Heures Creuses (00h-06h et 22h-24h)
ax2.axvspan(zoom_jour.index.min(), pd.to_datetime('2026-06-22 06:00:00+00:00'), color='#a8dadc', alpha=0.35, label='Heures Creuses (HC)')
ax2.axvspan(pd.to_datetime('2026-06-22 22:00:00+00:00'), zoom_jour.index.max(), color='#a8dadc', alpha=0.35)
ax2.set_title("2. Zoom sur 24h (Lundi 22 juin) : Nuit, réveil, repas de midi et dîner", fontweight='bold')
ax2.set_ylabel("Puissance (Watts)")
ax2.set_xlabel("Heure")
ax2.legend(loc='upper right')

# 3. Température extérieure sur ces mêmes 24h
ax3.plot(zoom_jour.index, zoom_jour['temperature_ext_C'], color='#e9c46a', linewidth=2.2, label='Température ext. (°C)')
ax3.set_title("3. Température extérieure sur la même journée (°C)", fontweight='bold')
ax3.set_ylabel("Température (°C)")
ax3.set_xlabel("Heure")
ax3.legend(loc='upper left')

plt.tight_layout()
plt.show()


### 🔍 Ce que vos yeux viennent déjà de découvrir :
1. **La puissance ne descend JAMAIS à zéro** : Même à 3 heures du matin quand tout le monde dort, la courbe oscille autour d'une ligne plancher verte à **~510 W**. C'est le **talon de veille permanent** (la box internet, la VMC, le frigo, les téléviseurs en veille).
2. **Des pics très intenses** : Vers 12h-13h et surtout 19h-21h, la puissance bondit soudainement à plus de **4 000 W** (plaques de cuisson, four, bouilloire, chauffe-eau).
3. **Le disjoncteur général** : La ligne pointillée rouge à **6 000 W** représente la puissance souscrite auprès du fournisseur (abonnement 6 kVA). Un dépassement prolongé ferait disjoncter toute la maison !
4. **Des plages tarifaires contrastées** : Les zones ombrées en bleu clair correspondent aux **Heures Creuses (HC)**, où l'électricité coûte moins cher qu'en **Heures Pleines (HP)**.

Maintenant que vous visualisez parfaitement le problème, apprenons à explorer, nettoyer et modéliser ces données avec **Pandas** !


---
## 1. Cadrage du problème, Objectifs & Échelles de mesure

### 🎯 Comprendre le problème avant d'écrire la moindre ligne de code
En Data Science, toute analyse commence par la **compréhension du domaine métier** (*Data & Problem Understanding*) :
1. **Quelles sont les questions analytiques ?**
   - La puissance appelée approche-t-elle la limite du disjoncteur général (**6 kVA = 6 000 W**) au risque de faire sauter le compteur ?
   - À quels moments de la journée et de la semaine consomme-t-on le plus ?
   - Quel est le coût financier réel de l'électricité consommée (€) ?
   - Quelle proportion de l'énergie est engloutie par le « talon de veille » (la consommation passive et permanente des appareils non éteints) ?
2. **Quelle est la variable cible vs les variables explicatives ?**
   - La **puissance active ($W$)** et l'**énergie ($kWh$)** sont les grandeurs centrales à mesurer, expliquer ou prédire.
   - L'heure de la journée, le jour de la semaine, la température extérieure et la plage tarifaire sont les variables explicatives (*features*).

---

### 🧭 Pourquoi le cadrage de Stevens est-il la boussole opérationnelle du Data Scientist ?
En 1946, le psychologue et statisticien Stanley Smith Stevens a formalisé les **4 échelles de mesure fondamentales** : *Nominale*, *Ordinale*, *Intervalle* et *Rapport (Ratio)*.

Loin d'être une simple classification théorique, ce cadrage est **le filtre de décision n°1** qui dicte chacun de vos choix techniques :

1. **Dans ce notebook (avec Pandas & Matplotlib)** : Il détermine quelles opérations mathématiques vous avez le droit d'effectuer (`mean()`, `median()`, `mode()`, `sum()`) et quel type de graphique tracer (diagramme en barres, histogramme de densité, série temporelle).
2. **Dans la suite du cours (en Machine Learning avec Scikit-Learn et PyTorch)** :
   - **Pour encoder vos variables d'entrée** : Les algorithmes (arbres de décision, Random Forest, réseaux de neurones) n'acceptent que des matrices de nombres. Si vous attribuez des entiers arbitraires `1, 2, 3` à une variable nominale (`salon`, `cuisine`, `buanderie`), un modèle linéaire ou neuronal croira faussement que `buanderie (3) > salon (1)` ! Stevens vous impose d'utiliser le **One-Hot Encoding** (`pd.get_dummies()`) pour le nominal, et l'**Ordinal Encoding** pour l'ordinal.
   - **Pour formuler votre problème de Machine Learning** :
     - Si la variable cible à prédire est **nominale** (ex: *« Quel appareil vient de s'allumer : Frigo, Four ou Lave-linge ? »*) $\rightarrow$ C'est un problème de **Classification**.
     - Si la variable cible est **de rapport** (ex: *« Quelle est la puissance exacte en Watts appelée à cet instant ? »*) $\rightarrow$ C'est un problème de **Régression**.

---

### 📊 Tableau de correspondance opérationnelle : Du capteur au Machine Learning

| Échelle | Définition & Propriétés | Exemple dans la maison | Ce que cela dicte dans Pandas (Tool 3) | Impact direct dans la suite (Machine Learning - Parties 1, 2, 3) |
|---|---|---|---|---|
| **Nominale** | Catégories qualitatives **sans aucun ordre hiérarchique** (étiquettes pures). | `circuit_dominant` (`cuisine`, `salon`), `periode_tarifaire` (`HP`, `HC`) | ❌ **Interdit** : Calculer une moyenne (la moyenne de « cuisine » et « salon » n'existe pas !)<br>✅ **Autorisé** : Comptages de fréquence (`value_counts()`), mode, diagrammes en barres (`bar`). | 🤖 **Encodage One-Hot obligatoire** (`pd.get_dummies`) pour ne pas créer un faux ordre numérique artificiel. Si c'est la cible à prédire $\rightarrow$ **Classification multi-classes** (NILM). |
| **Ordinale** | Catégories qualitatives possédant un **ordre hiérarchique**, mais sans distance métrique mesurable entre les niveaux. | `statut_capteur` (`Normal` < `Alerte_Pic`), `niveau_conso` (`Veille` < `Modéré` < `Élevé` < `Pic`) | ❌ **Interdit** : Opérations arithmétiques directes ($+, \times$).<br>✅ **Autorisé** : Comparaisons ($<, >$), médiane, percentiles (`quantile()`), discrétisation ordonnée avec `pd.cut()`. | 🤖 **Encodage Ordinal** (`0, 1, 2, 3`). Contrairement au nominal, conserver la relation $0 < 1 < 2 < 3$ est bénéfique car le modèle apprend qu'un pic est plus sévère qu'une veille. |
| **Intervalle** | Grandeurs numériques avec un intervalle régulier et mesurable, mais dont le **zéro est arbitraire** (convention humaine). | `temperature_ext_C` (0°C est le point de gel de l'eau, pas l'absence totale d'énergie thermique !) | ❌ **Interdit** : Ratios et pourcentages (dire « à 20°C il fait deux fois plus chaud qu'à 10°C » est un contresens physique !).<br>✅ **Autorisé** : Différences ($T_2 - T_1$), moyennes (`mean()`), écarts-types (`std()`). | 🤖 **Standardisation Z-Score** : On centre et réduit $(x - \mu)/\sigma$ pour stabiliser l'apprentissage des gradients sans supposer de zéro absolu. |
| **Rapport (Ratio)** | Grandeurs physiques continues possédant un **vrai zéro absolu** (zéro = absence totale de la grandeur). | `puissance_W`, `tension_V`, `energie_kWh`, `duree_s`, `cout_eur` | ❌ **Aucune interdiction**.<br>✅ **Autorisé** : Toutes les opérations mathématiques, ratios (4 kW = exactement 2× plus que 2 kW), conversions physiques ($W \rightarrow kW \rightarrow kWh$) et **sommes cumulées** (`resample('1D').sum()`). | 🤖 **Régression continue** : Prédire directement la valeur physique en Watts. Les normalisations Min-Max $[0, 1]$ et les ratios d'amplitude conservent toute leur signification physique. |

---

### ⚠️ Les 3 erreurs classiques évitées grâce à Stevens :
1. **L'erreur de la « moyenne absurde »** : Tenter d'agréger par moyenne une colonne textuelle ou un identifiant de capteur. Dans Pandas, `df.describe()` sépare d'ailleurs nativement les variables numériques (rapport/intervalle) des variables textuelles (`include='str'`).
2. **Le piège du codage en Machine Learning** : Si vous assignez arbitrairement `0 = salon`, `1 = cuisine`, `2 = garage`, une régression linéaire supposera que `garage = 2 × cuisine`, ce qui faussera tout l'apprentissage du modèle ! Stevens vous rappelle que le nominal exige du One-Hot.
3. **La confusion flux vs stock** : Sur une échelle de rapport, une grandeur de flux (puissance instantanée en $W$) se moyenne dans le temps (`resample('1h').mean()`), tandis qu'une grandeur de stock (énergie consommée en $kWh$) s'additionne (`resample('1D').sum()`).


In [ ]:
# Programme 3 — QCM 1
qcm(
    "Pourquoi la température en degrés Celsius (°C) appartient-elle à l'échelle d'intervalle et NON à l'échelle de rapport ?",
    [
        "Parce qu'elle ne peut pas être mesurée avec une virgule.",
        "Parce que le zéro (0°C) est une convention (température de fusion de l'eau) : 20°C ne représente pas 'deux fois plus chaud' que 10°C.",
        "Parce qu'il est interdit de calculer la moyenne de températures en Data Science.",
        "Parce que les degrés Celsius ne s'appliquent qu'à l'extérieur de la maison."
    ],
    correct=1,
    explication="Une grandeur appartient à l'échelle de rapport uniquement si son zéro est physique et absolu (comme le Kelvin pour la température, ou le Watt pour la puissance : 0 W signifie aucune énergie consommée). Sur une échelle d'intervalle comme le Celsius, le zéro est arbitraire."
)


---
## 2. Découverte de Pandas : Series, DataFrames & Diagnostic Initial

Pour utiliser Pandas, on importe conventionnellement la bibliothèque sous l'alias `pd` :
```python
import pandas as pd
```

### 🧱 Les deux structures reines de Pandas :
1. **`pd.Series`** : Un tableau à 1 dimension, étiqueté par un **Index**. C'est l'équivalent d'une colonne unique d'une table avec son en-tête.
2. **`pd.DataFrame`** : Une table à 2 dimensions composée de plusieurs colonnes (`Series`) partageant le même index. C'est l'équivalent d'une feuille de calcul ou d'une table de base de données SQL.

```
          Index       Colonne 1       Colonne 2          Colonne 3
            ↓             ↓               ↓                  ↓
          0       2026-06-20T15:56       5030.0             HP       ← Ligne (Row)
          1       2026-06-20T15:56       5030.0             HP
          2       2026-06-21T13:29        710.0             HP
```

### 📂 Charger des données depuis un fichier CSV :
Pandas fournit la fonction ultra-polyvalente `pd.read_csv()` :
```python
df = pd.read_csv("chemin/vers/fichier.csv")
```

### 🔍 Les méthodes incontournables d'inspection rapide :
- **`df.head(n)`** : Affiche les $n$ premières lignes (5 par défaut).
- **`df.tail(n)`** : Affiche les $n$ dernières lignes.
- **`df.shape`** : Tuple `(nombre_de_lignes, nombre_de_colonnes)`.
- **`df.columns`** : Liste des noms des colonnes.
- **`df.dtypes`** : Type de données de chaque colonne (`float64`, `int64`, `str`/`object`).
- **`df.info()`** : Vue d'ensemble concise (dimensions, types, mémoire, valeurs non-nulles).
- **`df.describe()`** : Résumé statistique complet des colonnes numériques (`count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`).
- **`df.describe(include='str')`** : Résumé des colonnes textuelles / catégorielles (`count`, `unique`, modalité la plus fréquente `top`, fréquence `freq`).

### 🎯 Sélectionner des données :
- Sélectionner **une seule colonne** (renvoie une `Series`) : `df['puissance_W']`
- Sélectionner **plusieurs colonnes** (renvoie un sous-`DataFrame`) : `df[['puissance_W', 'tension_V']]`

---

### ✏️ Exercice 1 — Diagnostic initial du dataset de la maison

Chargez le fichier de relevés `data/conso_maison_capteurs.csv` dans une variable `df`.

Ensuite, extrayez :
1. `type_df` : Le type Python de l'objet `df` (obtenu avec `type(df)`).
2. `shape_df` : Le tuple représentant les dimensions `(lignes, colonnes)`.
3. `colonnes_df` : La liste des noms des colonnes sous forme de liste standard (`list(df.columns)`).
4. `puissance_moyenne_brute` : La moyenne de la colonne `puissance_W` (`float(df['puissance_W'].mean())`).

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
df = pd.read_csv("data/conso_maison_capteurs.csv")
type_df = type(df)
shape_df = df.shape
colonnes_df = list(df.columns)
puissance_moyenne_brute = float(df['puissance_W'].mean())
```
</details>


In [ ]:
# Programme 4 — Démonstration : Chargement et inspection
df_demo = pd.read_csv("data/conso_maison_capteurs.csv")

print(f"Format de la table : {df_demo.shape[0]:,} lignes et {df_demo.shape[1]} colonnes.")
print("\n--- Aperçu des 3 premières observations ---")
display(df_demo.head(3))

print("\n--- Résumé univarié des variables numériques ---")
display(df_demo.describe())

print("\n--- Résumé univarié des variables textuelles/catégorielles ---")
display(df_demo.describe(include='str'))


In [ ]:
# Programme 5 — Exercice 1
# À toi de jouer : charge les données et renseigne les variables demandées :

df = pd.read_csv("data/conso_maison_capteurs.csv")

type_df = ...
shape_df = ...
colonnes_df = ...
puissance_moyenne_brute = ...


In [ ]:
# Programme 6 — Test Exercice 1
# Exécutez cette cellule pour valider vos réponses
check("type_df", type_df, pd.DataFrame)
check("shape_df", shape_df, (99396, 7))
check("colonnes_df", colonnes_df, ['timestamp', 'puissance_W', 'tension_V', 'temperature_ext_C', 'circuit_dominant', 'periode_tarifaire', 'statut_capteur'])
check("puissance_moyenne_brute", puissance_moyenne_brute, 1160.098, tol=1.0)


---
## 3. Nettoyage & Gestion des Données Manquantes (`NaN`)

### ❓ Qu'est-ce qu'un `NaN` ?
En Python et Pandas, une valeur manquante est représentée par l'objet spécial **`np.nan`** (*Not a Number*).
Contrairement à `0` ou `""` (chaîne vide), un `NaN` indique **l'absence totale de donnée**.

### 📡 D'où viennent les `NaN` dans les capteurs IoT ?
Dans un système domotique réel :
- **Pertes de paquets radio** : perturbations du signal Wi-Fi ou Zigbee entre la prise connectée et la passerelle.
- **Redémarrage de box ou micro-coupure** : pendant 3 minutes, aucun paquet n'est transmis.
- **Défaillance ponctuelle de capteur** : la tension réseau n'est pas reçue alors que la puissance l'est.

### 🔍 Détecter et compter les valeurs manquantes :
- **`df.isna()`** (ou son synonyme `df.isnull()`) : Renvoie un DataFrame de booléens (`True` si la case est manquante, `False` sinon).
- **`df.isna().sum()`** : Fait la somme des `True` par colonne, ce qui donne immédiatement le **nombre de valeurs manquantes par variable** !

### 🛠️ Comment traiter les valeurs manquantes en séries temporelles ?
1. **La suppression brute (`dropna()`)** : Dangereuse en séries temporelles ! Supprimer des lignes casse la régularité du temps (l'écart entre les lignes ne serait plus de 10 secondes fixes).
2. **Le remplacement par zéro (`fillna(0)`)** : **Erreur physique fatale !** Remplacer par 0 W ferait croire que le four et le frigo se sont éteints pendant la coupure Wi-Fi, sous-estimant la consommation réelle.
3. **La propagation vers l'avant (`ffill()` pour *forward-fill*)** : On propage la dernière mesure valide connue. Sur des relevés toutes les 10 secondes, la puissance 10 secondes plus tard est très probablement identique. C'est la solution de référence pour les brèves coupures de réseau !

---

### ✏️ Exercice 2 — Détection et imputation des trames perdues

1. Calculez le nombre de valeurs manquantes initiales dans la colonne `puissance_W` et stockez-le dans `nb_manquants_initial` (`int(df['puissance_W'].isna().sum())`).
2. Créez une copie propre `df_clean = df.copy()`.
3. Imputez les valeurs manquantes des colonnes `puissance_W` et `tension_V` avec la méthode `.ffill()` :
   `df_clean['puissance_W'] = df_clean['puissance_W'].ffill()`
   `df_clean['tension_V'] = df_clean['tension_V'].ffill()`
4. Vérifiez qu'il ne reste aucune valeur manquante dans `df_clean['puissance_W']` et stockez ce nombre dans `nb_manquants_apres` (doit valoir 0).

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
nb_manquants_initial = int(df['puissance_W'].isna().sum())
df_clean = df.copy()
df_clean['puissance_W'] = df_clean['puissance_W'].ffill()
df_clean['tension_V'] = df_clean['tension_V'].ffill()
nb_manquants_apres = int(df_clean['puissance_W'].isna().sum())
```
</details>


In [ ]:
# Programme 7 — QCM 2
qcm(
    "Pourquoi la méthode df['puissance_W'].fillna(0) est-elle une mauvaise pratique pour nettoyer les pertes réseau d'un compteur électrique ?",
    [
        "Parce que Pandas refuse techniquement d'insérer le nombre 0 dans une colonne de flottants.",
        "Parce qu'elle sous-estime l'énergie réelle consommée : la maison et ses appareils ont continué de fonctionner pendant la panne de transmission.",
        "Parce que la valeur 0 entraîne une division par zéro dans tous les calculs suivants.",
        "Parce qu'il est obligatoire d'utiliser la valeur maximale."
    ],
    correct=1,
    explication="Exactement ! Si votre box domotique perd la liaison radio pendant 5 minutes à midi alors que le four cuit votre repas, remplacer par 0 W efface l'électricité consommée et fausse le bilan énergétique. La propagation de la dernière valeur (ffill) ou l'interpolation reflète bien mieux la physique réelle."
)


In [ ]:
# Programme 8 — Démonstration : Détection des NaN et imputation
manquants_par_colonne = df.isna().sum()

print("📊 Bilan des valeurs manquantes dans le dataset :")
for col, val in manquants_par_colonne.items():
    pct = (val / len(df)) * 100
    if val > 0:
        print(f"  • Colonne '{col}' : {val} valeurs manquantes ({pct:.2f} %)")
    else:
        print(f"  • Colonne '{col}' : 0 valeur manquante (100 % complet)")


In [ ]:
# Programme 9 — Exercice 2
# À toi de jouer : détecte et impute les valeurs manquantes :

nb_manquants_initial = ...

df_clean = df.copy()
df_clean['puissance_W'] = ...
df_clean['tension_V'] = ...

nb_manquants_apres = ...


In [ ]:
# Programme 10 — Test Exercice 2
# Exécutez cette cellule pour valider vos réponses
check("nb_manquants_initial", nb_manquants_initial, 135)
check("nb_manquants_apres", nb_manquants_apres, 0)
check("df_clean sans NaN", int(df_clean['tension_V'].isna().sum()), 0)


---
## 4. Feature Engineering : Créer de nouvelles colonnes métier

Le **Feature Engineering** (ou ingénierie des variables) consiste à créer de nouvelles colonnes dérivées à partir des colonnes existantes pour mieux répondre aux questions métier.

Dans Pandas, créer une nouvelle colonne est aussi simple que d'écrire :
```python
df['nouvelle_colonne'] = ...
```

Dans notre cas énergétique, la puissance brute en Watts ne suffit pas. Nous avons besoin de **4 nouvelles variables métier** :

1. **`puissance_kW`** : La puissance active exprimée en kilowatts ($1\text{ kW} = 1000\text{ W}$) :
   $$\text{puissance\_kW} = \frac{\text{puissance\_W}}{1000}$$

2. **`energie_kWh`** : L'énergie consommée pendant l'intervalle de temps entre deux mesures (ici $\Delta t = 10\text{ secondes}$).
   Rappel de physique : $\text{Énergie (Joules)} = \text{Puissance (Watts)} \times \text{Temps (secondes)}$.
   Comme $1\text{ kWh} = 1000\text{ W} \times 3600\text{ s} = 3\,600\,000\text{ Joules}$ :
   $$\text{energie\_kWh} = \frac{\text{puissance\_W} \times 10}{3600 \times 1000}$$

3. **`niveau_conso` (Discrétisation ordinale avec `pd.cut`)** :
   La fonction `pd.cut()` permet de découper une variable continue en catégories ordonnées selon des seuils (*bins*) :
   - $< 400\text{ W}$ : `'Veille'` (consommation de base du logement)
   - $400\text{ à }1500\text{ W}$ : `'Modéré'` (activité normale, TV, éclairage, frigo)
   - $1500\text{ à }3500\text{ W}$ : `'Élevé'` (appareils de chauffe, four, lave-linge)
   - $> 3500\text{ W}$ : `'Pic'` (forte charge simultanée, risques de disjonction)

4. **`cout_eur`** : Le coût financier de chaque tranche de 10 secondes calculé selon la période tarifaire :
   - Heures Pleines ($HP$) : **0.27 € / kWh**
   - Heures Creuses ($HC$) : **0.20 € / kWh**

---

### ✏️ Exercice 3 — Calcul des grandeurs énergétiques et financières

Ajoutez à `df_clean` les 4 colonnes définies ci-dessus :
1. `puissance_kW`
2. `energie_kWh`
3. `niveau_conso` (utilisez `pd.cut(df_clean['puissance_W'], bins=[-np.inf, 400, 1500, 3500, np.inf], labels=['Veille', 'Modéré', 'Élevé', 'Pic'])`)
4. `cout_eur` (utilisez `np.where(df_clean['periode_tarifaire'] == 'HP', 0.27, 0.20) * df_clean['energie_kWh']`)

Calculez ensuite :
- `energie_totale_kWh` : La somme de l'énergie totale consommée sur toute la période.
- `facture_estimee_eur` : Le montant total estimé de la facture d'électricité (€).

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
df_clean['puissance_kW'] = df_clean['puissance_W'] / 1000.0
df_clean['energie_kWh'] = (df_clean['puissance_W'] * 10.0) / (3600.0 * 1000.0)

bins = [-np.inf, 400, 1500, 3500, np.inf]
labels = ['Veille', 'Modéré', 'Élevé', 'Pic']
df_clean['niveau_conso'] = pd.cut(df_clean['puissance_W'], bins=bins, labels=labels)

prix_kwh = np.where(df_clean['periode_tarifaire'] == 'HP', 0.27, 0.20)
df_clean['cout_eur'] = df_clean['energie_kWh'] * prix_kwh

energie_totale_kWh = float(df_clean['energie_kWh'].sum())
facture_estimee_eur = float(df_clean['cout_eur'].sum())
```
</details>


In [ ]:
# Programme 11 — QCM 3
qcm(
    "Un radiateur de 2 000 W fonctionne sans interruption pendant 30 minutes. Quelle quantité d'énergie a-t-il consommée en kilowattheures (kWh) ?",
    [
        "60 kWh",
        "2 kWh",
        "1 kWh",
        "2 000 kWh"
    ],
    correct=2,
    explication="2 000 W = 2 kW. L'énergie est le produit de la puissance par le temps : 2 kW × 0.5 h = 1 kWh. Ne confondez jamais la puissance instantanée (le débit d'un robinet en W) et l'énergie totale (le volume du seau rempli en kWh) !"
)


In [ ]:
# Programme 12 — Démonstration : Feature engineering avec Pandas
demo_mini = pd.DataFrame({
    'puissance_W': [300.0, 2200.0, 4800.0],
    'periode_tarifaire': ['HC', 'HP', 'HP']
})

demo_mini['puissance_kW'] = demo_mini['puissance_W'] / 1000.0
demo_mini['energie_kWh'] = (demo_mini['puissance_W'] * 10.0) / (3600.0 * 1000.0)
demo_mini['niveau_conso'] = pd.cut(demo_mini['puissance_W'], bins=[-np.inf, 400, 1500, 3500, np.inf], labels=['Veille', 'Modéré', 'Élevé', 'Pic'])
demo_mini['cout_eur'] = demo_mini['energie_kWh'] * np.where(demo_mini['periode_tarifaire'] == 'HP', 0.27, 0.20)

display(demo_mini)


In [ ]:
# Programme 13 — Exercice 3
# À toi de jouer : calcule les 4 nouvelles colonnes dans df_clean :

df_clean['puissance_kW'] = ...
df_clean['energie_kWh'] = ...
df_clean['niveau_conso'] = ...
df_clean['cout_eur'] = ...

energie_totale_kWh = ...
facture_estimee_eur = ...


In [ ]:
# Programme 14 — Test Exercice 3
# Exécutez cette cellule pour valider vos réponses
check("energie_totale_kWh", round(energie_totale_kWh, 1), 320.3, tol=0.5)
check("facture_estimee_eur", round(facture_estimee_eur, 2), 79.07, tol=0.2)
check("niveau_conso (catégories)", list(df_clean['niveau_conso'].cat.categories), ['Veille', 'Modéré', 'Élevé', 'Pic'])


---
## 5. DatetimeIndex & Slicing Temporel Express

Par défaut, l'index d'un DataFrame est une suite d'entiers `0, 1, 2, ...`.
Lorsque l'on travaille avec des données temporelles, le coup de génie de Pandas consiste à transformer la colonne temporelle en **`DatetimeIndex`** :

```python
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp')
```

### 🪄 Ce que permet un `DatetimeIndex` :
1. **Accéder à des propriétés temporelles instantanément** :
   - `df.index.hour` : L'heure de la journée ($0$ à $23$)
   - `df.index.day_name()` : Le nom du jour en anglais (*Monday, Tuesday...*)
   - `df.index.dayofweek` : Le jour de la semaine sous forme d'entier ($0 = \text{Lundi}, 6 = \text{Dimanche}$)
   - `df.index.dayofweek >= 5` : Détecte en un clin d'œil si c'est le week-end (Samedi ou Dimanche)
2. **Le filtrage direct par date (Slicing en langage naturel)** :
   - `df.loc['2026-06-25']` : Extrait directement toute la journée du 25 juin 2026 !
   - `df.loc['2026-06-21':'2026-06-23']` : Extrait une plage de dates continues.
3. **Isoler un créneau horaire récurrent avec `.between_time()`** :
   - `df.between_time('19:00', '22:00')` : Extrait le créneau du dîner (19h-22h) pour tous les jours du dataset !

---

### ✏️ Exercice 4 — Analyse de la pointe du soir du premier dimanche

1. Créez une copie `df_ts = df_clean.copy()`.
2. Convertissez la colonne `'timestamp'` en datetime et placez-la en index :
   `df_ts['timestamp'] = pd.to_datetime(df_ts['timestamp'])`
   `df_ts = df_ts.set_index('timestamp')`
3. Créez 3 colonnes calendaires :
   - `df_ts['heure'] = df_ts.index.hour`
   - `df_ts['jour_semaine'] = df_ts.index.day_name()`
   - `df_ts['est_weekend'] = df_ts.index.dayofweek >= 5`
4. Isolez dans la variable `creneau_dimanche` la soirée du premier dimanche **`2026-06-21` entre `19:00` et `22:00`** :
   `creneau_dimanche = df_ts.loc['2026-06-21'].between_time('19:00', '22:00')`
5. Calculez la puissance moyenne observée pendant ce créneau et stockez-la dans `puissance_moy_dimanche_pointe`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
df_ts = df_clean.copy()
df_ts['timestamp'] = pd.to_datetime(df_ts['timestamp'])
df_ts = df_ts.set_index('timestamp')

df_ts['heure'] = df_ts.index.hour
df_ts['jour_semaine'] = df_ts.index.day_name()
df_ts['est_weekend'] = df_ts.index.dayofweek >= 5

creneau_dimanche = df_ts.loc['2026-06-21'].between_time('19:00', '22:00')
puissance_moy_dimanche_pointe = float(creneau_dimanche['puissance_W'].mean())
```
</details>


In [ ]:
# Programme 15 — Démonstration : Indexation temporelle et slicing
df_demo_ts = df_clean.copy()
df_demo_ts['timestamp'] = pd.to_datetime(df_demo_ts['timestamp'])
df_demo_ts = df_demo_ts.set_index('timestamp')

print("Type de l'index :", type(df_demo_ts.index))
print("Début des relevés :", df_demo_ts.index.min())
print("Fin des relevés   :", df_demo_ts.index.max())

# Exemple d'extraction express : la matinée du 22 juin de 8h à 10h
extrait_matin = df_demo_ts.loc['2026-06-22 08:00':'2026-06-22 10:00']
print(f"Nombre de points relevés entre 8h et 10h le 22/06 : {len(extrait_matin)}")


In [ ]:
# Programme 16 — Exercice 4
# À toi de jouer : configure le DatetimeIndex et analyse le créneau de pointe :

df_ts = df_clean.copy()
df_ts['timestamp'] = ...
df_ts = ...

df_ts['heure'] = ...
df_ts['jour_semaine'] = ...
df_ts['est_weekend'] = ...

creneau_dimanche = ...
puissance_moy_dimanche_pointe = ...


In [ ]:
# Programme 17 — Test Exercice 4
# Exécutez cette cellule pour valider vos réponses
check("Type de l'index", isinstance(df_ts.index, pd.DatetimeIndex), True)
check("creneau_dimanche (lignes)", len(creneau_dimanche), 1008)
check("puissance_moy_dimanche_pointe", round(puissance_moy_dimanche_pointe, 1), 1208.0, tol=1.0)


---
## 6. Rééchantillonnage temporel (`resample`) & Bilans Journaliers

### ⏱️ Pourquoi le rééchantillonnage est la méthode reine en énergie
Dans le monde des capteurs, les données arrivent à haute fréquence (toutes les 10 secondes). Mais pour faire une facture ou étudier des tendances, on souhaite agréger ces données par heure (`'1h'`) ou par jour (`'1D'`).

La méthode **`.resample()`** est le `groupby` temporel de Pandas :
- `'1h'` : Agrège par blocs de 1 heure
- `'1D'` : Agrège par journée entière

### ⚠️ La Règle d'Or de l'Agrégation Énergétique :
- Pour une grandeur de **flux / débit instantané** (Puissance en $W$, Tension en $V$, Température en °C) : on calcule la **MOYENNE** (`.mean()`).
- Pour une grandeur de **stock / cumul** (Énergie en $kWh$, Facture en €) : on calcule la **SOMME** (`.sum()`).

```python
# Puissance moyenne horaire (W)
p_horaire = df_ts['puissance_W'].resample('1h').mean()

# Énergie quotidienne cumulée (kWh)
energie_jour = df_ts['energie_kWh'].resample('1D').sum()
```

---

### ✏️ Exercice 5 — Bilan quotidien et journée record

1. Construisez un DataFrame `df_journalier` rééchantillonné au pas journalier (`'1D'`) contenant :
   - La somme de `'energie_kWh'`
   - La somme de `'cout_eur'`
   - La moyenne de `'puissance_W'` (nommez cette colonne `'puissance_moy_W'`)
2. Identifiez la date du jour le plus énergivore de toute la période (au format chaîne `'YYYY-MM-DD'`, utilisez `.idxmax().strftime('%Y-%m-%d')` sur la colonne `energie_kWh`) et stockez-la dans `jour_record_conso`.
3. Stockez la valeur de cette consommation record ($kWh$) dans `conso_jour_record_kWh`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
df_journalier = df_ts[['energie_kWh', 'cout_eur']].resample('1D').sum()
df_journalier['puissance_moy_W'] = df_ts['puissance_W'].resample('1D').mean()

jour_record_conso = df_journalier['energie_kWh'].idxmax().strftime('%Y-%m-%d')
conso_jour_record_kWh = float(df_journalier['energie_kWh'].max())
```
</details>


In [ ]:
# Programme 18 — QCM 4
qcm(
    "Quelle est la différence fondamentale entre df.groupby('heure')['puissance_W'].mean() et df['puissance_W'].resample('1h').mean() ?",
    [
        "Aucune, ce sont deux écritures strictement équivalentes qui donnent le même résultat.",
        "groupby('heure') regroupe tous les jours ensemble pour donner une journée type de 24 heures (0h à 23h), alors que resample('1h') conserve la chronologie continue (chaque heure de chaque jour reste distincte).",
        "resample supprime automatiquement les valeurs manquantes alors que groupby les conserve.",
        "groupby ne peut s'appliquer que sur des nombres entiers."
    ],
    correct=1,
    explication="Très important ! resample('1h') préserve la flèche du temps (série temporelle continue heure par heure sur 2 semaines), tandis que groupby('heure') superpose toutes les journées pour dégager le comportement moyen à chaque heure de la journée."
)


In [ ]:
# Programme 19 — Démonstration : Rééchantillonnage avec resample()
demo_bilan = df_ts[['energie_kWh', 'cout_eur']].resample('1D').sum().head(3)
print("Aperçu du bilan d'énergie et de coût pour les 3 premiers jours :")
display(demo_bilan)


In [ ]:
# Programme 20 — Exercice 5
# À toi de jouer : calcule le bilan quotidien et identifie le jour record :

df_journalier = ...
df_journalier['puissance_moy_W'] = ...

jour_record_conso = ...
conso_jour_record_kWh = ...


In [ ]:
# Programme 21 — Test Exercice 5
# Exécutez cette cellule pour valider vos réponses
check("jour_record_conso", jour_record_conso, "2026-07-01")
check("conso_jour_record_kWh", round(conso_jour_record_kWh, 1), 43.3, tol=0.5)
check("df_journalier (colonnes)", list(df_journalier.columns), ['energie_kWh', 'cout_eur', 'puissance_moy_W'])


---
## 7. Fenêtres Glissantes (`rolling`) & Talon de Veille

### 🌊 Lisser les signaux avec la moyenne mobile
Le signal brut de puissance mesuré toutes les 10 secondes présente d'incessantes micro-variations.
La méthode **`.rolling(window=N)`** fait glisser une fenêtre de $N$ points et calcule une statistique locale (moyenne, écart-type, minimum).
- Avec une mesure toutes les 10 secondes, une fenêtre de **60 points** correspond à un lissage sur $60 \times 10\text{ s} = 600\text{ s} = 10\text{ minutes}$.

### 🔌 Qu'est-ce que le « Talon de Veille » (Base Load) ?
Même en pleine nuit quand tous les occupants dorment, la puissance d'une maison ne tombe jamais à 0 Watt. Elle oscille autour d'une valeur plancher appelée **talon de veille** :
- Box internet + décodeur : ~25 W
- VMC (Ventilation permanente) : ~35 W
- Réfrigérateur (au repos / compresseur) : ~80 W
- Appareils en veille (TV, chargeurs branchés, électroménager) : ~50 à 100 W

Le talon moyen d'un foyer se situe généralement entre **300 W et 600 W**. Sur une année, un talon permanent de 500 W représente une dépense passive d'environ **1 000 € par an** !

En Data Science, on estime ce talon de veille de façon robuste grâce au **5ᵉ percentile** (`df['puissance_W'].quantile(0.05)`).

---

### ✏️ Exercice 6 — Lissage et estimation du talon de veille

1. Calculez dans `df_ts['puissance_lissee_W']` la moyenne mobile sur une fenêtre de **60 points** avec `min_periods=1` :
   `df_ts['puissance_W'].rolling(window=60, min_periods=1).mean()`.
2. Estimez le talon de veille permanent de la maison en calculant le **5ᵉ percentile** de la puissance (`float(df_ts['puissance_W'].quantile(0.05))`) et stockez-le dans `talon_veille_W`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
df_ts['puissance_lissee_W'] = df_ts['puissance_W'].rolling(window=60, min_periods=1).mean()
talon_veille_W = float(df_ts['puissance_W'].quantile(0.05))
```
</details>


In [ ]:
# Programme 22 — Démonstration : Rolling window et détection du bruit de fond
zoom = df_ts.loc['2026-06-21 11:00':'2026-06-21 15:00']

plt.figure(figsize=(11, 4))
plt.plot(zoom.index, zoom['puissance_W'], color='#adb5bd', alpha=0.6, label="Signal brut (10 s)")
plt.plot(zoom.index, zoom['puissance_W'].rolling(60).mean(), color='#e63946', linewidth=2, label="Lissage mobile (10 min)")
plt.title("Lissage du signal de puissance par fenêtre glissante (rolling mean)")
plt.ylabel("Puissance (Watts)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Programme 23 — Exercice 6
# À toi de jouer : calcule la puissance lissée et le talon de veille :

df_ts['puissance_lissee_W'] = ...
talon_veille_W = ...


In [ ]:
# Programme 24 — Test Exercice 6
# Exécutez cette cellule pour valider vos réponses
check("talon_veille_W", round(talon_veille_W, 0), 510.0, tol=10.0)
check("puissance_lissee_W (sans NaN)", int(df_ts['puissance_lissee_W'].isna().sum()), 0)


---
## 8. Profilage Métier & Tableau de Bord Visuel

Pandas s'intègre parfaitement avec **Matplotlib** pour transformer des données brutes en indicateurs visuels percutants :
1. **Répartition du temps par régime de charge** : Combien de temps la maison passe-t-elle en veille vs en pic ?
2. **Courbe de charge moyenne sur 24 heures** : Comparaison du rythme de vie en semaine (Lundi-Vendredi) vs le week-end (Samedi-Dimanche).
3. **Bilan énergétique quotidien empilé** : Part Heures Pleines vs Heures Creuses jour par jour.
4. **Distribution statistique de la puissance** : Histogramme positionné par rapport au disjoncteur général (**6 000 W**).


In [ ]:
# Programme 25 — Visualisation complète : Tableau de bord de la consommation du foyer
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Bar chart des niveaux de consommation
comptages = df_ts['niveau_conso'].value_counts()[['Veille', 'Modéré', 'Élevé', 'Pic']]
couleurs_cat = ['#2a9d8f', '#e9c46a', '#f4a261', '#e76f51']
axes[0, 0].bar(comptages.index, comptages.values, color=couleurs_cat, edgecolor='black', alpha=0.85)
axes[0, 0].set_title("1. Répartition du temps par régime de consommation", fontweight='bold')
axes[0, 0].set_ylabel("Nombre de relevés (tranches de 10 s)")
for i, v in enumerate(comptages.values):
    axes[0, 0].text(i, v + 1500, f"{v/len(df_ts)*100:.1f} %", ha='center', fontweight='bold')

# 2. Courbe de charge moyenne sur 24 heures (Semaine vs Week-end)
profil_semaine = df_ts[~df_ts['est_weekend']].groupby('heure')['puissance_W'].mean()
profil_weekend = df_ts[df_ts['est_weekend']].groupby('heure')['puissance_W'].mean()
axes[0, 1].plot(profil_semaine.index, profil_semaine.values, marker='o', color='#1d3557', linewidth=2.5, label="Semaine (Lun-Ven)")
axes[0, 1].plot(profil_weekend.index, profil_weekend.values, marker='s', color='#e63946', linewidth=2.5, linestyle='--', label="Week-end (Sam-Dim)")
axes[0, 1].axhline(talon_veille_W, color='green', linestyle=':', label=f"Talon veille (~{talon_veille_W:.0f} W)")
axes[0, 1].set_title("2. Courbe de charge horaire moyenne (Profil 24h)", fontweight='bold')
axes[0, 1].set_xlabel("Heure de la journée (0h à 23h)")
axes[0, 1].set_ylabel("Puissance moyenne (W)")
axes[0, 1].set_xticks(range(0, 24, 2))
axes[0, 1].legend()

# 3. Énergie quotidienne (Heures Pleines vs Heures Creuses)
bilan_hp_hc = df_ts.groupby([df_ts.index.date, 'periode_tarifaire'])['energie_kWh'].sum().unstack()
bilan_hp_hc.plot(kind='bar', stacked=True, color=['#457b9d', '#e63946'], ax=axes[1, 0], edgecolor='black', alpha=0.85)
axes[1, 0].set_title("3. Bilan énergétique quotidien (HP vs HC)", fontweight='bold')
axes[1, 0].set_ylabel("Énergie consommée (kWh)")
axes[1, 0].set_xlabel("Jour")
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Distribution de la puissance active
axes[1, 1].hist(df_ts['puissance_W'], bins=60, color='#3a86ff', edgecolor='black', alpha=0.75, range=(0, 6000))
axes[1, 1].axvline(df_ts['puissance_W'].mean(), color='red', linestyle='--', linewidth=2, label=f"Moyenne ({df_ts['puissance_W'].mean():.0f} W)")
axes[1, 1].axvline(6000, color='darkred', linestyle='-', linewidth=2.5, label="Disjoncteur 6 kVA (6 000 W)")
axes[1, 1].set_title("4. Distribution de la puissance (Histogramme)", fontweight='bold')
axes[1, 1].set_xlabel("Puissance active (Watts)")
axes[1, 1].set_ylabel("Fréquence")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


---
## 9. 🏆 Défi Final — L'Audit Énergétique de la Maison

Vous êtes mandaté en tant que **Data Scientist** pour produire le rapport d'audit énergétique de ce foyer sur les 2 semaines de relevés.
Mobilisez tout ce que vous avez appris pour calculer les 4 indicateurs clés :

1. **`facture_totale_eur`** : Quel est le montant exact arrondi au centime (€) de la facture d'électricité sur les 2 semaines ? (`round(float(df_ts['cout_eur'].sum()), 2)`).
2. **`part_heures_creuses_pct`** : Quel est le pourcentage (%) de l'énergie totale consommée pendant les **Heures Creuses** ?
   $$\text{part\_HC} = \frac{\sum \text{energie\_kWh}_{HC}}{\sum \text{energie\_kWh}_{totale}} \times 100$$
3. **`heure_pic_max`** : À quelle heure de la journée ($0$ à $23$) la puissance moyenne du foyer atteint-elle son pic maximal ? (utilisez `int(df_ts.groupby('heure')['puissance_W'].mean().idxmax())`).
4. **`energie_veille_totale_kWh`** : Sachant que le talon de veille est de `talon_veille_W` (environ 510 W) permanent sur toute la durée couverte (durée en heures : $\text{nb\_lignes} \times 10 / 3600$), quelle quantité d'énergie ($kWh$) a été consommée par cette veille passive ?
   $$\text{energie\_veille} = \frac{\text{talon\_veille\_W} \times \text{duree\_totale\_heures}}{1000}$$

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
facture_totale_eur = round(float(df_ts['cout_eur'].sum()), 2)

energie_hc = df_ts[df_ts['periode_tarifaire'] == 'HC']['energie_kWh'].sum()
energie_totale = df_ts['energie_kWh'].sum()
part_heures_creuses_pct = round(float((energie_hc / energie_totale) * 100), 2)

heure_pic_max = int(df_ts.groupby('heure')['puissance_W'].mean().idxmax())

duree_totale_heures = (len(df_ts) * 10.0) / 3600.0
energie_veille_totale_kWh = round(float((talon_veille_W * duree_totale_heures) / 1000.0), 2)
```
</details>


In [ ]:
# Programme 26 — Défi Final
# À toi de jouer : calcule les 4 indicateurs de l'audit énergétique :

facture_totale_eur = ...
part_heures_creuses_pct = ...
heure_pic_max = ...
energie_veille_totale_kWh = ...


In [ ]:
# Programme 27 — Test Défi Final
# Exécutez cette cellule pour valider l'ensemble de votre audit énergétique !
check("1. Facture totale (€)", facture_totale_eur, 79.07, tol=0.2)
check("2. Part Heures Creuses (%)", part_heures_creuses_pct, 33.06, tol=0.5)
check("3. Heure de pic maximum (h)", heure_pic_max, 21)
check("4. Énergie talon de veille (kWh)", energie_veille_totale_kWh, 140.81, tol=2.0)


---
## 🎓 Synthèse du Section 5 : Ce que vous maîtrisez désormais avec Pandas

Félicitations ! Vous venez de franchir une étape déterminante dans votre parcours de Data Scientist :

1. **La structure d'un jeu de données tabulaire** : Vous savez manipuler les `Series` et les `DataFrames`, les inspecter avec `.head()`, `.shape`, `.info()`, et calculer des résumés statistiques numériques et textuels avec `.describe()`.
2. **Les 4 échelles de mesure** : Vous savez distinguer les données nominales, ordinales, d'intervalle et de rapport pour choisir les bonnes transformations mathématiques.
3. **Le nettoyage des données IoT** : Vous comprenez la nature des valeurs manquantes (`NaN`) et savez pourquoi la propagation vers l'avant (`ffill`) est la méthode adaptée pour combler les coupures de transmission radio sans biaiser la physique du problème.
4. **L'ingénierie des variables (*Feature Engineering*)** : Vous savez créer de nouvelles colonnes, convertir des unités ($W \rightarrow kW \rightarrow kWh$), découper en classes ordinales avec `pd.cut()` et chiffrer des coûts avec des conditions tarifaires.
5. **L'indexation temporelle (`DatetimeIndex`)** : Vous maîtrisez le découpage par dates (`.loc['2026-06-25']`) et par créneaux horaires réguliers (`.between_time()`).
6. **Le rééchantillonnage temporel (`resample`)** : Vous savez agréger des séries haute fréquence au pas horaire ou journalier en respectant la règle d'or : **moyenne** pour les grandeurs de flux (puissance), **somme** pour les grandeurs de stock (énergie, coût).
7. **Le lissage et la charge de fond (`rolling`)** : Vous savez éliminer le bruit haute fréquence et quantifier le gaspillage permanent du talon de veille.

---

### 🚀 Prochaine étape :
Dans les modules suivants du projet (**Section 7 — Le Réfrigérateur**, **Partie 2 — Signatures Fréquentielles & Features**, et **Partie 3 — La Maison Complète**), vous mobiliserez ces compétences Pandas pour découper les signatures d'appareils électriques et entraîner des modèles de Machine Learning (Random Forest, DTW, CNN) pour désagréger la consommation (*NILM — Non-Intrusive Load Monitoring*).
